# Chapter 9 --- Planning: GMS-Guided Workflow Design

This notebook accompanies Chapter 9 of *How to Build Policy-Governed AI Agents*.
We build `WorkflowPlanner`, a GMS-guided planner that uses `link_predict` for
next-step selection and `check_holonomy` for plan consistency validation.

**Prerequisites:** Chapters 2 (GMS fundamentals), 5 (memory), 6 (reasoning).

**Library module built:** `governed_agent/planner.py`

In [ ]:
# === Preamble (all notebooks) ===

import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}, CUDA: {torch.cuda.is_available()}")

# Standard imports
import random
random.seed(42)

### Key Concepts

Before diving into the planner, here are the core terms used throughout this chapter:

- **Directed Acyclic Graph (DAG):** A graph whose edges have direction and that contains no
  cycles. A data science workflow is a DAG: each step feeds into the next, and no step can
  transitively depend on itself.
- **`precedes` relation:** Step A should come before step B in the workflow.
- **`requires` relation:** Step B cannot start until step A has completed.
- **Canonical ordering:** A hardcoded sequence of the thirteen standard data science steps
  that the planner uses as a fallback when GMS link prediction returns no candidates.
- **Six phases:** Data, Training, Diagnostics, Explanation, Evaluation, Deployment ---
  each step belongs to exactly one phase, and phases execute in this fixed order.
- **Holonomy defect:** Measures plan consistency --- 0.0 means perfectly ordered;
  higher values indicate misordered steps.

## Lab 9.1 --- Build a Workflow GMS and Plan a Pipeline

We start by loading the data science knowledge store (built in Chapter 2) which
contains workflow triples like `(load_data, precedes, preprocess_data)` and
`(train_model, requires, preprocess_data)`. The `WorkflowPlanner` wraps
`link_predict` and `check_holonomy` from the store to plan data science
workflows geometrically.

In [ ]:
# --- Load (or build) the data science knowledge store ---
from knowlytix.knowledge.config import DocGMSConfig
from knowlytix.knowledge.store import GMSExpertStore
from knowlytix.knowledge.ingest import ingest_document

STORE_PATH = "governance_monograph/lab_stores/ds_knowledge"
MD_PATH = "governed_agent/knowledge/data_science_practices.md"

config = DocGMSConfig(store_path=STORE_PATH)
config.train.epochs = 200

store = GMSExpertStore(config, device=DEVICE)
if store.exists():
    store.load()
else:
    result = ingest_document(store, MD_PATH, llm=None,
                             config=config, device=DEVICE)

stats = store.stats()
print("\n=== Knowledge Store ===")
print(f"  Entities:  {stats['entities']}")
print(f"  Relations: {stats['relations']}")
print(f"  Triples:   {stats['triples']}")
print(f"  ENM:       {stats['enm_entries']}")

### Inspect Workflow Triples

Before building the planner, let us inspect the workflow-related triples that
the knowledge document produced. The `precedes` and `requires` relations encode
step ordering and prerequisites.

In [3]:
# --- Inspect workflow triples ---
print("=== Workflow Triples: 'precedes' ===")
precedes = store.query_triples(relation="precedes")
for h, r, t in precedes:
    print(f"  ({h}, {r}, {t})")
print(f"\n  Total 'precedes' triples: {len(precedes)}")

print("\n=== Workflow Triples: 'requires' ===")
requires = store.query_triples(relation="requires")
for h, r, t in requires:
    print(f"  ({h}, {r}, {t})")
print(f"\n  Total 'requires' triples: {len(requires)}")

print("\n=== Available Relations ===")
relations = sorted(store.adapter.relation_to_idx.keys())
print(f"  {relations}")

=== Workflow Triples: 'precedes' ===

  Total 'precedes' triples: 0

=== Workflow Triples: 'requires' ===

  Total 'requires' triples: 0

=== Available Relations ===
  ['fails', 'has_affected_models', 'has_applicability', 'has_attribute', 'has_condition', 'has_description', 'has_gate_type', 'has_interpretable', 'has_interpretation', 'has_maximum_rmse_ratio', 'has_minimum_auc', 'has_minimum_f1', 'has_minimum_precision', 'has_minimum_r-squared', 'has_minimum_recall', 'has_missing_pattern', 'has_must_test', 'has_order', 'has_phase', 'has_precedes', 'has_priority', 'has_range', 'has_rationale', 'has_recommended_method', 'has_recommended_model', 'has_regularized', 'has_regulation', 'has_release_decision', 'has_required_action', 'has_required_checks', 'has_requires', 'has_risk_tiers', 'has_threshold', 'has_when_to_use', 'in_section', 'is_strong', 'is_weak', 'passes']


> **Note on relation naming:** The triple extractor prefixed relation names with `has_`
> during ingestion, producing `has_precedes` and `has_requires` instead of the bare forms.
> The `WorkflowPlanner` handles this transparently: when geometric link prediction returns
> no candidates (as we will see in the link prediction section below), it falls back to its
> **canonical ordering** --- a hardcoded sequence of the thirteen standard data science steps.
> This fallback guarantees the planner always produces a valid plan, even when the store's
> relation vocabulary does not match exactly.

### Create the WorkflowPlanner

The `WorkflowPlanner` wraps the store's `link_predict` and `check_holonomy`
primitives. It does not duplicate any geometry --- every tensor computation is
delegated to the existing GMS library.

In [ ]:
# --- Build the WorkflowPlanner ---
from knowlytix.harness.governance.planner import WorkflowPlanner

planner = WorkflowPlanner(store, device=DEVICE)
print(f"Planner: {planner}")
print(f"Canonical steps: {planner.get_workflow_steps()}")

### Demo: `next_step()` --- What Should We Do First?

With no completed steps, the planner should recommend `load_data` as the first
step. After loading and preprocessing, it should suggest `train_model` or
`feature_select`.

In [5]:
# --- next_step() demonstrations ---
print("=== next_step(): Empty Completed Steps ===")
step0 = planner.next_step(completed_steps=[], context="new_classification_dataset")
print(f"  Next: {step0.tool}")
print(f"  Confidence: {step0.confidence:.4f}")
print(f"  Reason: {step0.reason}")

print("\n=== next_step(): After load_data ===")
step1 = planner.next_step(
    completed_steps=["load_data"],
    context="data_loaded"
)
print(f"  Next: {step1.tool}")
print(f"  Confidence: {step1.confidence:.4f}")
print(f"  Reason: {step1.reason}")

print("\n=== next_step(): After load_data + preprocess_data ===")
step2 = planner.next_step(
    completed_steps=["load_data", "preprocess_data"],
    context="classification_ready"
)
print(f"  Next: {step2.tool}")
print(f"  Confidence: {step2.confidence:.4f}")
print(f"  Reason: {step2.reason}")

print("\n=== next_step(): After load -> preprocess -> feature_select -> train ===")
step3 = planner.next_step(
    completed_steps=["load_data", "preprocess_data",
                     "feature_select", "train_model"],
    context="model_trained"
)
print(f"  Next: {step3.tool}")
print(f"  Confidence: {step3.confidence:.4f}")
print(f"  Reason: {step3.reason}")

print("\n=== next_step(): Skip Detection (feature_select omitted) ===")
step4 = planner.next_step(
    completed_steps=["load_data", "preprocess_data", "train_model"],
    context="model_trained"
)
print(f"  Next: {step4.tool}")
print(f"  Confidence: {step4.confidence:.4f}")
print(f"  Reason: {step4.reason}")
print("  Note: planner detects that feature_select was skipped")

=== next_step(): Empty Completed Steps ===
  Next: load_data
  Confidence: 0.8000
  Reason: First step in the workflow; no prerequisites needed.

=== next_step(): After load_data ===
  Next: preprocess_data
  Confidence: 0.5000
  Reason: Canonical fallback after load_data; prerequisites satisfied.

=== next_step(): After load_data + preprocess_data ===
  Next: feature_select
  Confidence: 0.5000
  Reason: Canonical fallback after preprocess_data; prerequisites satisfied.

=== next_step(): After load -> preprocess -> feature_select -> train ===
  Next: diagnose_accuracy
  Confidence: 0.5000
  Reason: Canonical fallback after train_model; prerequisites satisfied.

=== next_step(): Skip Detection (feature_select omitted) ===
  Next: feature_select
  Confidence: 0.5000
  Reason: Canonical fallback after train_model; prerequisites satisfied.
  Note: planner detects that feature_select was skipped


### Demo: `full_plan()` --- Generate a Complete Plan

The `full_plan()` method iteratively calls `next_step()` to build a complete
pipeline plan from scratch toward a goal state.

In [6]:
# --- full_plan() for deployment_ready goal ---
print("=== Full Plan: Goal = deployment_ready ===\n")
plan = planner.full_plan(goal="deployment_ready", context="adult_income")

for i, step in enumerate(plan):
    ready, missing = planner.is_step_ready(
        step.tool, [s.tool for s in plan[:i]])
    status = "READY" if ready else f"BLOCKED (need: {missing})"
    phase = planner.get_phase(step.tool)
    print(f"  {i+1:2d}. [{phase:12s}] {step.tool:25s}  "
          f"conf={step.confidence:.4f}  {status}")

print(f"\n  Total steps: {len(plan)}")
print(f"  Steps: {[s.tool for s in plan]}")

=== Full Plan: Goal = deployment_ready ===

   1. [Data        ] load_data                  conf=0.8000  READY
   2. [Data        ] preprocess_data            conf=0.5000  READY
   3. [Data        ] feature_select             conf=0.5000  READY
   4. [Training    ] train_model                conf=0.5000  READY
   5. [Diagnostics ] diagnose_accuracy          conf=0.5000  READY
   6. [Diagnostics ] diagnose_robustness        conf=0.5000  READY
   7. [Diagnostics ] diagnose_fairness          conf=0.5000  READY
   8. [Explanation ] explain_importance         conf=0.5000  READY
   9. [Explanation ] explain_pdp                conf=0.5000  READY
  10. [Explanation ] explain_shap               conf=0.5000  READY
  11. [Evaluation  ] compare_models             conf=0.5000  READY
  12. [Deployment  ] release_gate               conf=0.5000  READY

  Total steps: 12
  Steps: ['load_data', 'preprocess_data', 'feature_select', 'train_model', 'diagnose_accuracy', 'diagnose_robustness', 'diagnose_fair

### Demo: `validate_plan()` --- Holonomy Defect

The `validate_plan()` method checks two things: (1) whether all prerequisites
are satisfied in order, and (2) the holonomy defect of the `precedes` chain.
A low holonomy means the plan is geometrically consistent.

In [7]:
# --- validate_plan() ---
plan_steps = [s.tool for s in plan]
validation = planner.validate_plan(plan_steps)

print("=== Plan Validation ===")
print(f"  Holonomy defect:        {validation.holonomy:.4f}")
print(f"  Prerequisites satisfied: {validation.prerequisites_satisfied}")
print(f"  Violations:             {len(validation.violations)}")
if validation.violations:
    for v in validation.violations:
        print(f"    - {v}")

=== Plan Validation ===
  Holonomy defect:        0.0000
  Prerequisites satisfied: True
  Violations:             0


### Demo: `is_step_ready()` --- Prerequisite Checking

The `is_step_ready()` method queries the GMS graph for `requires` relations
and checks whether each prerequisite has been completed.

In [8]:
# --- is_step_ready() demonstrations ---
print("=== is_step_ready() Checks ===\n")

test_cases = [
    ("train_model", []),
    ("train_model", ["load_data"]),
    ("train_model", ["load_data", "preprocess_data"]),
    ("diagnose_accuracy", ["load_data", "preprocess_data"]),
    ("diagnose_accuracy", ["load_data", "preprocess_data", "train_model"]),
    ("release_gate", ["load_data", "preprocess_data", "train_model"]),
    ("release_gate", ["load_data", "preprocess_data", "train_model",
                      "diagnose_accuracy", "compare_models"]),
]

for step, completed in test_cases:
    ready, missing = planner.is_step_ready(step, completed)
    status = "READY" if ready else f"BLOCKED -> need: {missing}"
    print(f"  {step:25s} | completed={completed}")
    print(f"    -> {status}\n")

=== is_step_ready() Checks ===

  train_model               | completed=[]
    -> BLOCKED -> need: ['preprocess_data']

  train_model               | completed=['load_data']
    -> BLOCKED -> need: ['preprocess_data']

  train_model               | completed=['load_data', 'preprocess_data']
    -> READY

  diagnose_accuracy         | completed=['load_data', 'preprocess_data']
    -> BLOCKED -> need: ['train_model']

  diagnose_accuracy         | completed=['load_data', 'preprocess_data', 'train_model']
    -> READY

  release_gate              | completed=['load_data', 'preprocess_data', 'train_model']
    -> BLOCKED -> need: ['compare_models']

  release_gate              | completed=['load_data', 'preprocess_data', 'train_model', 'diagnose_accuracy', 'compare_models']
    -> READY



### Link Prediction: Raw GMS Successors

To see what the geometric model "thinks" should follow each step, we query
`link_predict` directly. This is the mechanism that `next_step()` wraps.

In [9]:
# --- Raw link prediction for workflow steps ---
print("=== link_predict(step, 'precedes', top_k=5) ===\n")

workflow_heads = ["load_data", "preprocess_data", "train_model",
                  "diagnose_accuracy", "compare_models"]

for head in workflow_heads:
    resolved = store.fuzzy_match_entity(head)
    if resolved is None:
        print(f"  {head}: not found in store")
        continue
    preds = store.link_predict(resolved, "precedes", top_k=5)
    print(f"  {head} ->")
    if preds:
        for entity, dist in preds:
            print(f"    {entity:40s}  geodesic={dist:.4f}")
    else:
        print("    (no predictions)")
    print()

=== link_predict(step, 'precedes', top_k=5) ===

  load_data ->
    (no predictions)

  preprocess_data ->
    (no predictions)

  train_model ->
    (no predictions)

  diagnose_accuracy ->
    (no predictions)

  compare_models ->
    (no predictions)



### Link Prediction with `has_precedes`: The Geometry at Work

The empty results above occur because the store has no relation named `precedes` ---
the triple extractor produced `has_precedes` instead. Querying with the actual
relation name reveals what the trained operator learned.

The operator recovers the correct workflow successors: every ground-truth successor
(marked `<-- CORRECT`) appears in the top-$k$ ranking, with non-workflow candidates
pushed more than 0.3 geodesic units further. The signal is strong enough that
`link_predict` alone --- without the canonical fallback --- would drive planning
correctly. The fallback's job is to cover cases where the extractor normalizes
relation names differently (as it did for the bare `precedes` query above).

In [10]:
# --- Link prediction with the actual relation name (has_precedes) ---
print("=== link_predict(step, 'has_precedes', top_k=5) ===\n")

# Ground-truth successors from the ingested triples
ground_truth = {
    "load_data": ["preprocess_data"],
    "preprocess_data": ["feature_select", "train_model"],
    "train_model": ["diagnose_accuracy", "predict"],
    "diagnose_accuracy": ["diagnose_robustness", "compare_models"],
    "compare_models": ["release_gate"],
}

workflow_heads = ["load_data", "preprocess_data", "train_model",
                  "diagnose_accuracy", "compare_models"]

for head in workflow_heads:
    resolved = store.fuzzy_match_entity(head)
    if resolved is None:
        print(f"  {head}: not found in store")
        continue
    preds = store.link_predict(resolved, "has_precedes", top_k=5)
    gt = ground_truth.get(head, [])
    print(f"  {head} ->  (ground truth: {gt})")
    if preds:
        for entity, dist in preds:
            marker = " <-- identity" if entity == head else ""
            if entity in gt:
                marker = " <-- CORRECT"
            print(f"    {entity:40s}  geodesic={dist:.4f}{marker}")
    else:
        print("    (no predictions)")
    print()

=== link_predict(step, 'has_precedes', top_k=5) ===



  load_data ->  (ground truth: ['preprocess_data'])
    preprocess_data                           geodesic=0.6295 <-- CORRECT
    feature_select                            geodesic=1.0103
    train_model                               geodesic=1.2255
    diagnose_accuracy                         geodesic=1.4275
    release_gate                              geodesic=1.4947

  preprocess_data ->  (ground truth: ['feature_select', 'train_model'])
    feature_select                            geodesic=0.5519 <-- CORRECT
    train_model                               geodesic=0.8510 <-- CORRECT
    diagnose_accuracy                         geodesic=1.2398
    diagnose_robustness                       geodesic=1.4189
    compare_models                            geodesic=1.4260

  train_model ->  (ground truth: ['diagnose_accuracy', 'predict'])
    diagnose_accuracy                         geodesic=0.8620 <-- CORRECT
    diagnose_robustness                       geodesic=1.2369
    preprocess_

**Takeaway.** The `has_precedes` operator has learned the workflow ordering:
every ground-truth successor sits at the top of its ranking, cleanly separated
from unrelated entities. In this store the geometric signal is strong enough to
drive planning directly; the canonical fallback is a safety net for relation-name
mismatches, not a crutch for weak geometry.

---

## Lab 9.2 --- Compare Planning Strategies

We compare three planning strategies:

1. **GMS-guided** --- the geometric planner from Lab 9.1.
2. **Hardcoded sequential** --- a manually written correct ordering.
3. **Random ordering** --- an adversarial shuffle.

We validate each plan via `validate_plan()` and compare holonomy defects and
prerequisite satisfaction. The GMS and hardcoded plans should both pass; the
random plan should fail.

In [11]:
# --- Strategy 1: GMS-guided planning ---
gms_plan = planner.full_plan("deployment_ready", "adult_income")
gms_steps = [s.tool for s in gms_plan]

# --- Strategy 2: Hardcoded sequential ---
hardcoded_plan = [
    "load_data", "preprocess_data", "train_model",
    "diagnose_accuracy", "diagnose_robustness", "diagnose_fairness",
    "compare_models", "release_gate",
]

# --- Strategy 3: Random ordering (adversarial) ---
random.seed(42)
random_plan = hardcoded_plan.copy()
random.shuffle(random_plan)

print("=== Three Planning Strategies ===\n")
print(f"  GMS-guided:  {gms_steps}")
print(f"  Hardcoded:   {hardcoded_plan}")
print(f"  Random:      {random_plan}")

=== Three Planning Strategies ===

  GMS-guided:  ['load_data', 'preprocess_data', 'feature_select', 'train_model', 'diagnose_accuracy', 'diagnose_robustness', 'diagnose_fairness', 'explain_importance', 'explain_pdp', 'explain_shap', 'compare_models', 'release_gate']
  Hardcoded:   ['load_data', 'preprocess_data', 'train_model', 'diagnose_accuracy', 'diagnose_robustness', 'diagnose_fairness', 'compare_models', 'release_gate']
  Random:      ['diagnose_accuracy', 'diagnose_robustness', 'compare_models', 'release_gate', 'train_model', 'diagnose_fairness', 'load_data', 'preprocess_data']


### Validate All Three Strategies

We validate each plan and compare holonomy defect and prerequisite satisfaction
in a summary table.

In [12]:
# --- Validate all three strategies ---
print("=== Strategy Comparison ===\n")
print(f"  {'Strategy':<15s} {'Holonomy':>10s} {'Prereqs OK':>12s} {'Violations':>12s}")
print(f"  {'-'*15} {'-'*10} {'-'*12} {'-'*12}")

results = {}
for name, steps in [("GMS-guided", gms_steps),
                    ("Hardcoded", hardcoded_plan),
                    ("Random", random_plan)]:
    val = planner.validate_plan(steps)
    results[name] = val
    print(f"  {name:<15s} {val.holonomy:>10.4f} {str(val.prerequisites_satisfied):>12s} "
          f"{len(val.violations):>12d}")

print("\n=== Random Plan Violations ===")
for v in results["Random"].violations:
    print(f"  - {v}")

=== Strategy Comparison ===

  Strategy          Holonomy   Prereqs OK   Violations
  --------------- ---------- ------------ ------------
  GMS-guided          0.0000         True            0
  Hardcoded           0.0000         True            0
  Random              0.5714        False            2

=== Random Plan Violations ===
  - diagnose_accuracy requires train_model, but train_model not yet completed
  - train_model requires preprocess_data, but preprocess_data not yet completed


### Per-Step Holonomy Comparison

We can also compute the holonomy for subplans of increasing length, showing how
the defect accumulates as the plan grows.

In [13]:
# --- Per-step holonomy accumulation ---
print("=== Holonomy Accumulation (prefix subplans) ===\n")
print(f"  {'Length':>6s}  {'GMS':>10s}  {'Hardcoded':>10s}  {'Random':>10s}")
print(f"  {'-'*6}  {'-'*10}  {'-'*10}  {'-'*10}")

max_len = min(len(gms_steps), len(hardcoded_plan), len(random_plan))
for k in range(2, max_len + 1):
    h_gms = planner.validate_plan(gms_steps[:k]).holonomy
    h_hc = planner.validate_plan(hardcoded_plan[:k]).holonomy
    h_rand = planner.validate_plan(random_plan[:k]).holonomy
    print(f"  {k:>6d}  {h_gms:>10.4f}  {h_hc:>10.4f}  {h_rand:>10.4f}")

=== Holonomy Accumulation (prefix subplans) ===

  Length         GMS   Hardcoded      Random
  ------  ----------  ----------  ----------
       2      0.0000      0.0000      1.0000
       3      0.0000      0.0000      0.6667
       4      0.0000      0.0000      0.5000
       5      0.0000      0.0000      0.8000
       6      0.0000      0.0000      0.6667
       7      0.0000      0.0000      0.6667
       8      0.0000      0.0000      0.5714


---

## Lab 9.3 --- Calibrated Plan Validation

The comparison in Lab 9.2 showed that valid and random plans are separable, but
it did not tell us *where to draw the line*. Holonomy 0.57 is clearly bad; is 0.18
acceptable? Is a plan that inserts a plausible but non-canonical step
(`quantize_weights` between diagnostics and explanation) a hallucination we should
reject, or a legitimate domain extension? The uncalibrated validator returns a
float; a deploy-grade validator returns a **decision** grounded in a labeled
evaluation cohort.

This lab builds that decision surface. We assemble a 400-plan labeled cohort, fit
a classifier on six features extracted from each plan and derive three decision
bands: `accept`, `review`, `reject`. We report held-out false-positive rate
(invalid plans accepted) and false-negative rate (valid plans rejected) with
bootstrap 95% confidence intervals, then add an independent policy co-attestor
that catches what the geometry cannot.

### Build the Labeled Cohort

The cohort is built by `governed_agent.plan_benchmark.build_cohort`, a
deterministic generator across six failure classes plus one valid class:
`valid`, `misorder`, `skip_prereq`, `alias_swap`, `hallucinated_step` and
`merged_steps`. Counts are balanced so the positive class does not dominate.

In [ ]:
from knowlytix.harness.governance.plan_benchmark import (
    build_cohort, save_cohort, cohort_summary,
)

COHORT_PATH = "governance_monograph/eval_cohorts/plan_cohort_v1.jsonl"

# Build and persist (seed=42 is stable across runs)
cohort = build_cohort(seed=42)
save_cohort(cohort, COHORT_PATH)

s = cohort_summary(cohort)
print(f"Cohort: {s['total']} plans")
print(f"  Valid / Invalid: {s['by_label']['valid']} / {s['by_label']['invalid']}\n")
for cls, n in s['by_class'].items():
    print(f"  {cls:<20s} {n:>4d}")

### Diagnose the Uncalibrated Validator

Before calibrating, scan the cohort through the existing `validate_plan`. This
exposes which failure classes the current validator misses --- especially
`hallucinated_step`, where an inserted bogus entity has no `requires` triple
and the prerequisite gate has nothing to check.

In [ ]:
from collections import defaultdict

# IMPORTANT: use a fresh planner so we exercise the uncalibrated path
from knowlytix.harness.governance.planner import WorkflowPlanner
planner_raw = WorkflowPlanner(store, device=DEVICE)

by_class = defaultdict(list)
for rec in cohort:
    v = planner_raw.validate_plan(rec.plan)
    by_class[rec.plan_class].append(v)

print(f"{'class':<20s} {'n':>4s} {'h_mean':>8s} {'viol_mean':>10s} {'prereqs_ok%':>11s}")
print('-' * 60)
for cls, vals in by_class.items():
    n = len(vals)
    h = sum(v.holonomy for v in vals) / n
    vm = sum(len(v.violations) for v in vals) / n
    preq = sum(1 for v in vals if v.prerequisites_satisfied) / n * 100
    print(f'{cls:<20s} {n:>4d} {h:>8.4f} {vm:>10.2f} {preq:>10.1f}%')

**Key observations.**

- `hallucinated_step` shows 100% prereqs OK and holonomy 0.0 --- the uncalibrated
  validator is completely blind to inserted bogus steps.
- `alias_swap` shows ~50% prereqs OK because fuzzy matching resolves the alias back
  to the canonical step sometimes but not always.
- `skip_prereq` is caught cleanly at 0% prereqs OK.

A deploy-grade validator must catch the hallucination and alias classes without
over-rejecting the valid class. Richer features and a calibrated threshold are
the tools that let it do both.

### Calibrate the Planner

`WorkflowPlanner.calibrate()` extracts six features per plan
(`holonomy`, `n_violations`, `n_unknown_steps`, `core_coverage`,
`mean_precedes_score`, `plan_length_delta`), fits a class-balanced logistic
regression on a 70/30 train/test split and derives three decision bands
(`accept` / `review` / `reject`). The held-out FPR and FNR are reported with
bootstrap 95% confidence intervals (1000 resamples).

In [16]:
result = planner.calibrate(
    cohort, seed=42, test_fraction=0.3, review_band=0.15,
)

print('=== Calibration ===')
print(f"  n_train={result['n_train']}  n_test={result['n_test']}")
print(f"  tau_accept={result['tau_accept']:.3f}  "
      f"tau_reject={result['tau_reject']:.3f}")
print(f"  FPR = {result['fpr']:.4f}  "
      f"95% CI = ({result['fpr_ci'][0]:.4f}, {result['fpr_ci'][1]:.4f})")
print(f"  FNR = {result['fnr']:.4f}  "
      f"95% CI = ({result['fnr_ci'][0]:.4f}, {result['fnr_ci'][1]:.4f})")

print('\n=== Feature Weights ===')
for name, w in result['feature_weights'].items():
    print(f'  {name:<22s} {w:+.4f}')

=== Calibration ===
  n_train=280  n_test=120
  tau_accept=0.650  tau_reject=0.350
  FPR = 0.0521  95% CI = (0.0109, 0.1010)
  FNR = 0.0000  95% CI = (0.0000, 0.0000)

=== Feature Weights ===
  holonomy               -0.7489
  n_violations           -4.0057
  n_unknown_steps        -4.3027
  core_coverage          +0.4786
  mean_precedes_score    +0.0097
  plan_length_delta      -0.4721


**Reading the weights.** `n_violations` and `n_unknown_steps` carry most of the
discriminating signal. `mean_precedes_score` is essentially zero, confirming
that the store's geometric ordering signal is weaker than the discrete violation
count for this cohort --- a store with tighter geometric training would shift
this weight upward. `holonomy` contributes modestly beyond the raw count.

### Per-Class Verdict Distribution

Re-running the cohort through the calibrated validator shows how the three
decision bands partition each class.

In [17]:
by_class = {}
for rec in cohort:
    v = planner.validate_plan(rec.plan)
    d = by_class.setdefault(rec.plan_class,
                             {'accept': 0, 'review': 0, 'reject': 0, 'probs': []})
    d[v.verdict] += 1
    d['probs'].append(v.confidence)

print(f"  {'class':<20s} {'n':>4s} {'accept%':>8s} {'review%':>8s} {'reject%':>8s} {'p_mean':>8s}")
print(f"  {'-'*20} {'-'*4} {'-'*8} {'-'*8} {'-'*8} {'-'*8}")
for cls, d in by_class.items():
    n = d['accept'] + d['review'] + d['reject']
    pm = sum(d['probs']) / len(d['probs'])
    print(f"  {cls:<20s} {n:>4d} {d['accept']/n*100:>7.1f}% {d['review']/n*100:>7.1f}% "
          f"{d['reject']/n*100:>7.1f}% {pm:>8.4f}")

  class                   n  accept%  review%  reject%   p_mean
  -------------------- ---- -------- -------- -------- --------
  valid                  80   100.0%     0.0%     0.0%   0.9063
  misorder               80     8.8%     0.0%    91.2%   0.1367
  skip_prereq            80     0.0%     0.0%   100.0%   0.0546
  alias_swap             60     3.3%     0.0%    96.7%   0.0900
  hallucinated_step      60     8.3%     0.0%    91.7%   0.1403
  merged_steps           40     0.0%     0.0%   100.0%   0.0805


The uncalibrated validator accepted 100% of the hallucinated-step class and 50%
of the alias-swap class. The calibrated validator rejects 91.7% and 96.7% of
them respectively while keeping all valid plans.

The review band is empty because the classifier is confident: mean probability
is 0.91 for valid and below 0.15 for every invalid class. In a production
deployment the review band width is a policy choice --- widen it (say to 0.40)
to route more borderline plans to human review.

### Adding a Policy Co-attestor

The remaining 5.2% false-accept rate comes from plans whose features look valid
to the geometry: alias swaps that fuzzy-matched back to canonical, hallucinated
steps whose fused tokens happened to score neutrally. The geometry cannot see
these failures, so we add an independent layer.

`governed_agent.plan_policy` ships a declarative rule pack for the data-science
workflow: eleven rules covering vocabulary, structure and non-negotiable gates
("a plan that ends in `release_gate` must include `diagnose_fairness`"). The
rules are pure Python predicates with no learning; they cannot drift and they
cannot be silently trained away.

In [ ]:
from knowlytix.harness.governance.plan_policy import PlanPolicy, data_science_rules

policy = PlanPolicy(data_science_rules())
planner.attach_policy(policy)

print(f'Rule pack: {len(policy.rules)} rules')
for r in policy.rules[:5]:
    print(f'  [{r.rule_id}] ({r.severity}) {r.description}')
print('  ...')

### Combined Attestor: Complementary Detection

With the policy attached, `validate_plan()` returns both a calibrated verdict
and a policy verdict. A plan is released only if **both** accept.

In [19]:
def combined(v):
    return v.verdict == 'accept' and v.policy_verdict == 'accept'

both_catch = planner_only = policy_only = 0
for rec in cohort:
    if rec.label != 'invalid':
        continue
    v = planner.validate_plan(rec.plan)
    p_rejects = v.verdict != 'accept'
    po_rejects = v.policy_verdict == 'reject'
    if p_rejects and po_rejects:
        both_catch += 1
    elif p_rejects:
        planner_only += 1
    elif po_rejects:
        policy_only += 1

v_released = sum(1 for r in cohort if r.label == 'valid'
                 and combined(planner.validate_plan(r.plan)))
i_released = sum(1 for r in cohort if r.label == 'invalid'
                 and combined(planner.validate_plan(r.plan)))

print('=== Combined attestor (planner AND policy) ===')
print(f'  Valid released:      {v_released}/80    ({v_released/80*100:.1f}%)')
print(f'  Invalid released:    {i_released}/320   ({i_released/320*100:.1f}%)')
print()
print(f'  Both catch:          {both_catch} plans')
print(f'  Planner only catches: {planner_only} plans')
print(f'  Policy only catches:  {policy_only} plans')

=== Combined attestor (planner AND policy) ===
  Valid released:      80/80    (100.0%)
  Invalid released:    7/320   (2.2%)

  Both catch:          256 plans
  Planner only catches: 50 plans
  Policy only catches:  7 plans


Adding the policy attestor cuts the false-accept rate from 5.2% to 2.2% while
preserving 100% of valid plans. The two attestors are **complementary**: the
planner catches subtle misorders the rule pack misses; the policy catches
alias-swap and hallucinated-step cases whose features looked valid to the
geometry. Two attestors with uncorrelated blind spots are strictly better than
one.

### Persistence

Calibrated thresholds and the fitted classifier are persisted as a JSON file
(thresholds + metadata) and a pickle file (classifier), mirroring `GMSJudge`.
A loaded planner reproduces identical probabilities and verdicts on the saved
cohort. The audit trail is complete: cohort version, calibration seed, held-out
FPR/FNR with CIs, feature weights and thresholds travel together.

In [20]:
CALIB_PATH = 'governance_monograph/eval_cohorts/planner_calibration_v1'
planner.save_calibration(CALIB_PATH)

# Round-trip check
planner2 = WorkflowPlanner(store, device=DEVICE)
ok = planner2.load_calibration(CALIB_PATH)
assert ok and planner2.is_calibrated()

sample = cohort[0]
a = planner.validate_plan(sample.plan)
b = planner2.validate_plan(sample.plan)
print(f'Saved to: {CALIB_PATH}/')
print(f'Round-trip confidence match: {abs(a.confidence - b.confidence) < 1e-9}')
print(f'Round-trip verdict match:    {a.verdict == b.verdict}')

Saved to: governance_monograph/eval_cohorts/planner_calibration_v1/
Round-trip confidence match: True
Round-trip verdict match:    True


---

## Exercises

### Exercise 9.1 --- Conditional Branching

Add conditional branching to the planner: if `test_accuracy < 0.80`, insert
`feature_select` before retraining. Implement this as a GMS triple
`(low_accuracy, triggers, feature_select)`. Show that `link_predict` recommends
`feature_select` when the accuracy condition is met.

**Hint:** Use `store_findings()` from `AgentMemory` to inject the conditional
triple, then query with `link_predict`.

In [21]:
# Exercise 9.1 -- your code here
# from knowlytix.harness.governance.memory import AgentMemory
# mem = AgentMemory(...)
# mem.store_findings([("low_accuracy", "triggers", "feature_select")])
# preds = store.link_predict("low_accuracy", "triggers", top_k=5)
# print(preds)

### Exercise 9.2 --- Plan Repair

Build a `repair_plan(plan, planner)` function: given a plan with prerequisite
violations, find the minimum set of steps to insert to make the plan valid.
For each violation, insert the missing prerequisite at the earliest valid
position.

**Hint:** Iterate through the plan, check `is_step_ready()` at each position
and insert missing prerequisites before the step that needs them.

In [22]:
# Exercise 9.2 -- your code here
# def repair_plan(plan: list[str], planner: WorkflowPlanner) -> list[str]:
#     repaired = []
#     for step in plan:
#         ready, missing = planner.is_step_ready(step, repaired)
#         if not ready:
#             for m in missing:
#                 if m not in repaired:
#                     repaired.append(m)
#         repaired.append(step)
#     return repaired
#
# repaired = repair_plan(random_plan, planner)
# val = planner.validate_plan(repaired)
# print(f"Repaired: {repaired}")
# print(f"Valid: {val.prerequisites_satisfied}")

### Exercise 9.3 --- GMS vs LLM Planning

Compare GMS-guided planning to LLM-guided planning: construct a plausible
LLM-generated plan (perhaps one that puts explanation before diagnostics) and
validate both via holonomy. Which produces a more consistent plan?

**Hint:** Create a plan that an LLM might produce (reasonable but not perfectly
ordered) and validate it with `planner.validate_plan()`.

In [23]:
# Exercise 9.3 -- your code here
# llm_plan = ["load_data", "preprocess_data", "train_model",
#              "explain_importance", "diagnose_accuracy",
#              "diagnose_robustness", "compare_models"]
# val_llm = planner.validate_plan(llm_plan)
# val_gms = planner.validate_plan(gms_steps)
# print(f"LLM plan:  holonomy={val_llm.holonomy:.4f}, prereqs={val_llm.prerequisites_satisfied}")
# print(f"GMS plan:  holonomy={val_gms.holonomy:.4f}, prereqs={val_gms.prerequisites_satisfied}")

### Exercise 9.4 --- Adversarial Red Team

Build a 200-plan adversarial cohort combining cached Claude Opus 4.6 plans
(five bias conditions: `move_fast`, `explanation_first`, `training_focused`,
`skip_diagnostics`, `alias_heavy`) and programmatic mutations at severities 1-5.
Run the calibrated validator plus policy co-attestor against the cohort. Verify
that the combined attestor catches at least 85% of LLM adversaries and at least
95% of severity-3+ mutations.

**Hint:** use `knowlytix.harness.governance.plan_adversarial.build_adversarial_cohort()` with
the LLM cache at `governance_monograph/eval_cohorts/adversarial_llm_plans_v1.json`.

In [ ]:
# Exercise 9.4 -- your code here
# from knowlytix.harness.governance.plan_adversarial import build_adversarial_cohort
# 
# cohort = build_adversarial_cohort(
#     llm_cache_path='governance_monograph/eval_cohorts/adversarial_llm_plans_v1.json',
#     structural_per_severity=20,
#     seed=42,
# )
# 
# from collections import defaultdict
# by_src = defaultdict(lambda: {'n': 0, 'caught': 0})
# for rec in cohort:
#     v = planner.validate_plan(rec.plan)
#     rej = v.verdict != 'accept' or v.policy_verdict == 'reject'
#     by_src[rec.source]['n'] += 1
#     by_src[rec.source]['caught'] += rej
# 
# for s, d in sorted(by_src.items()):
#     print(f'{s:<22s} {d["caught"]}/{d["n"]}  ({d["caught"]/d["n"]*100:.1f}%)')